# Deploy ZeroEntropy's zembed-1 Model Package from AWS Marketplace


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook.

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

---


**NOTE: If you're not running this notebook on SageMaker Notebooks or SageMaker Studio, define `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`, and `SAGEMAKER_EXECUTION_ROLE_ARN` before launching the notebook server.**

zembed-1 is ZeroEntropy's flagship embedding model for high-accuracy semantic retrieval. It converts queries and documents into vectors for RAG, agents, semantic search, hybrid retrieval, clustering, deduplication, and knowledge-base indexing.

This sample notebook shows you how to deploy ZeroEntropy zembed-1 from AWS Marketplace using Amazon SageMaker.

> **Note**: This is a reference notebook. If you are not running in the same AWS Region as the model package ARN shown below, copy the Product ARN for your selected Region from the AWS Marketplace configuration page and replace `model_package_arn`.

## Pre-requisites:
1. **Note**: This notebook contains elements which render correctly in the Jupyter interface. Open this notebook from an Amazon SageMaker Notebook Instance or Amazon SageMaker Studio.
1. Ensure that the IAM role used has **AmazonSageMakerFullAccess** or equivalent SageMaker permissions.
1. To deploy this ML model successfully, ensure that:
    1. Either your IAM role has these three permissions and you have authority to make AWS Marketplace subscriptions in the AWS account used:
        1. **aws-marketplace:ViewSubscriptions**
        1. **aws-marketplace:Unsubscribe**
        1. **aws-marketplace:Subscribe**
    2. or your AWS account has a subscription to ZeroEntropy zembed-1. If so, skip step: [Subscribe to the model package](#1.-Subscribe-to-the-model-package)

## Contents:
1. [Subscribe to the model package](#1.-Subscribe-to-the-model-package)
2. [Create an endpoint and perform real-time inference](#2.-Create-an-endpoint-and-perform-real-time-inference)
   1. [Create an endpoint](#A.-Create-an-endpoint)
   2. [Create input payload](#B.-Create-input-payload)
   3. [Perform real-time inference](#C.-Perform-real-time-inference)
   4. [Visualize output](#D.-Visualize-output)
   5. [Delete the endpoint](#E.-Delete-the-endpoint)
3. [Perform batch inference](#3.-Perform-batch-inference)
4. [Clean-up](#4.-Clean-up)
    1. [Delete the model](#A.-Delete-the-model)
    2. [Unsubscribe to the listing (optional)](#B.-Unsubscribe-to-the-listing-(optional))

## Usage instructions
You can run this notebook one cell at a time by using Shift+Enter.


## 1. Subscribe to the model package


To subscribe to the model package:
1. Open the ZeroEntropy zembed-1 model package listing page in AWS Marketplace.
1. On the AWS Marketplace listing, click on the **Continue to subscribe** button.
1. On the **Subscribe to this software** page, review the EULA, pricing, and support terms, then click **Accept Offer** if you and your organization agree.
1. Click **Continue to configuration**, choose a **Region**, and copy the displayed **Product ARN**. This is the model package ARN that you need when creating a deployable model with Boto3.
1. Paste the Product ARN into the following cell if it differs from the default ARN.


In [ ]:
model_package_arn = "arn:aws:sagemaker:us-east-1:148761660947:model-package/zembed-1-v1-0"


## Install Required Dependencies

Before running this notebook, install the required packages:

```bash
# If using pip
pip install sagemaker boto3 numpy

# If using uv
uv add sagemaker boto3 numpy

# If using conda
conda install -c conda-forge sagemaker boto3 numpy
```

If you're running this in Amazon SageMaker Studio or a SageMaker Notebook Instance, most of these packages are pre-installed. You may only need to upgrade them:

```bash
pip install --upgrade sagemaker boto3
```

Or run this cell first:


In [ ]:
%pip install --upgrade sagemaker boto3 numpy


## Configure AWS Credentials

This section works without changes inside SageMaker Studio or a SageMaker Notebook Instance. If you run locally, set AWS credentials and `SAGEMAKER_EXECUTION_ROLE_ARN` before starting Jupyter.


In [ ]:
import os

import boto3
import numpy as np
import sagemaker as sage
from botocore.exceptions import ClientError
from sagemaker import get_execution_role

boto_session = boto3.Session()
region = boto_session.region_name or os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
sagemaker_session = sage.Session(boto_session=boto_session)
sm_client = boto_session.client("sagemaker", region_name=region)
runtime = boto_session.client("sagemaker-runtime", region_name=region)

try:
    role = get_execution_role()
except Exception:
    role = os.environ.get("SAGEMAKER_EXECUTION_ROLE_ARN")

if not role:
    raise ValueError(
        "Could not determine a SageMaker execution role. "
        "Run this notebook in SageMaker Studio/Notebook Instance or set SAGEMAKER_EXECUTION_ROLE_ARN."
    )

bucket = sagemaker_session.default_bucket()
print(f"Region: {region}")
print(f"Default bucket: {bucket}")
print(f"Execution role: {role}")


In [ ]:
import json
import time
import uuid
from pathlib import Path

model_package_region = model_package_arn.split(":")[3]
if model_package_region != region:
    print(
        f"This notebook is running in {region}, but model_package_arn is in {model_package_region}. "
        "Use the Product ARN for your selected AWS Marketplace Region or switch your notebook Region."
    )


## 2. Create an endpoint and perform real-time inference


If you want to understand how real-time inference with Amazon SageMaker works, see [Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-hosting.html).


In [ ]:
# Identifier for the endpoint for your organization. Add a short random suffix to avoid name collisions.
suffix = uuid.uuid4().hex[:8]
model_name = f"zembed-1-model-{suffix}"
endpoint_config_name = f"zembed-1-config-{suffix}"
endpoint_name = f"zembed-1-endpoint-{suffix}"

content_type = "application/json"
accept = "application/json"

real_time_inference_instance_type = (
    "ml.g6e.xlarge"  # Recommended for real-time inference, subject to availability in your AWS Region.
)
batch_transform_inference_instance_type = (
    "ml.g5.2xlarge"  # Recommended for batch transform, subject to availability in your AWS Region.
)

# zembed-1 uses CUDA 12-capable SageMaker AMIs for GPU deployment.
real_time_inference_ami_version = "al2-ami-sagemaker-inference-gpu-2-1"
batch_transform_ami_version = "al2-ami-sagemaker-batch-gpu-535"


### A. Create an endpoint

This usually takes 5-10 minutes.


In [ ]:
# Create a deployable SageMaker model from the AWS Marketplace model package.
sm_client.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={"ModelPackageName": model_package_arn},
)

sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InitialInstanceCount": 1,
            "InstanceType": real_time_inference_instance_type,
            "InitialVariantWeight": 1.0,
            "InferenceAmiVersion": real_time_inference_ami_version,
        }
    ],
)

sm_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)

waiter = sm_client.get_waiter("endpoint_in_service")
waiter.wait(EndpointName=endpoint_name)
print(f"Endpoint is in service: {endpoint_name}")


Once the endpoint has been created, you can perform real-time embedding requests.


In [ ]:
endpoint_name


### B. Create input payload

The input to zembed-1 is an `embedding_type` and one or more strings to embed.

```json
{
  "embedding_type": "query",
  "input": [
    "<string>"
  ],
  "dimensions": 1280
}
```

Use `"query"` for search queries and `"document"` for corpus passages. `dimensions` is optional. Supported values are `40`, `80`, `160`, `320`, `640`, `1280`, and `2560`.


In [ ]:
query_payload = {
    "embedding_type": "query",
    "input": ["what is the first step in making apple jam"],
    "dimensions": 1280,
}

document_payload = {
    "embedding_type": "document",
    "input": [
        "apple stocks were down 1%",
        "boil apples before adding sugar to make jam",
        "freeze apples for a cold dessert",
        "one rotten apple ruins the basket",
        "the preserve must be left to cool",
        "jam to some tunes",
    ],
    "dimensions": 1280,
}

print(json.dumps(query_payload, indent=2))


### C. Perform real-time inference


In [ ]:
def invoke_zembed(payload):
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType=content_type,
        Accept=accept,
        Body=json.dumps(payload).encode("utf-8"),
    )
    result_body = response["Body"].read()
    return json.loads(result_body.decode("utf-8"))

query_result = invoke_zembed(query_payload)
document_result = invoke_zembed(document_payload)

query_result


### D. Visualize output

zembed-1 returns an object containing `embeddings`, per-input token counts, and usage metadata. The vectors are returned in the same order as the input strings.


In [ ]:
query_embedding = np.array(query_result["embeddings"][0])
document_embeddings = np.array(document_result["embeddings"])

print(f"Query embeddings: {len(query_result['embeddings'])}")
print(f"Document embeddings: {len(document_result['embeddings'])}")
print(f"Vector dimension: {len(query_embedding)}")
print(f"Query token counts: {query_result.get('num_tokens')}")
print(f"Document token counts: {document_result.get('num_tokens')}")
print(f"Usage: {document_result.get('usage')}")


In [ ]:
# Rank documents by dot-product similarity to the query.
scores = document_embeddings @ query_embedding
ranked = sorted(enumerate(scores), key=lambda item: item[1], reverse=True)

print(f"Query: {query_payload['input'][0]}\n")
for rank, (idx, score) in enumerate(ranked, start=1):
    print(f"{rank}. score={score:.4f} | {document_payload['input'][idx]}")


### E. Delete the endpoint

Now that you have successfully performed real-time inference, you can terminate the endpoint to avoid being charged.


In [ ]:
sm_client.delete_endpoint(EndpointName=endpoint_name)

waiter = sm_client.get_waiter("endpoint_deleted")
waiter.wait(EndpointName=endpoint_name)

sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
print(f"Deleted endpoint and endpoint config: {endpoint_name}")


You can also invoke the endpoint with the AWS CLI:

```bash
aws sagemaker-runtime invoke-endpoint \
  --endpoint-name <endpoint-name> \
  --body fileb://input.json \
  --content-type application/json \
  --accept application/json \
  --region <region> \
  output.json
```


## 3. Perform batch inference


In this section, you will perform batch inference using multiple JSONL payloads. If you are not familiar with batch transform, see these links:
1. [How it works](https://docs.aws.amazon.com/sagemaker/latest/dg/ex1-batch-transform.html)
2. [How to run a batch transform job](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-batch.html)


In [ ]:
# If you deleted the real-time endpoint above, the SageMaker model still exists and can be reused for batch transform.
batch_input_filename = "zembed_batch_input.jsonl"
batch_records = [
    {
        "embedding_type": "query",
        "input": ["what is apple jam made from"],
        "dimensions": 1280,
    },
    {
        "embedding_type": "document",
        "input": [
            "Apple jam is made by cooking apples with sugar until the fruit thickens.",
            "A GPU instance can host a SageMaker endpoint for real-time inference.",
        ],
        "dimensions": 1280,
    },
]

with open(batch_input_filename, "w", encoding="utf-8") as f:
    for record in batch_records:
        f.write(json.dumps(record) + "\n")

print(Path(batch_input_filename).read_text())


In [ ]:
# Upload the batch-transform job input file to S3.
transform_input = sagemaker_session.upload_data(
    batch_input_filename,
    bucket=bucket,
    key_prefix=f"zembed-1/batch-input/{suffix}",
)
print("Transform input uploaded to " + transform_input)


In [ ]:
transform_job_name = f"zembed-1-transform-{suffix}"
transform_output = f"s3://{bucket}/zembed-1/batch-output/{suffix}/"

sm_client.create_transform_job(
    TransformJobName=transform_job_name,
    ModelName=model_name,
    TransformInput={
        "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": transform_input}},
        "ContentType": content_type,
        "SplitType": "Line",
    },
    TransformOutput={"S3OutputPath": transform_output, "Accept": accept},
    TransformResources={
        "InstanceType": batch_transform_inference_instance_type,
        "InstanceCount": 1,
        "TransformAmiVersion": batch_transform_ami_version,
    },
)

waiter = sm_client.get_waiter("transform_job_completed_or_stopped")
waiter.wait(TransformJobName=transform_job_name)
print(f"Transform job completed: {transform_job_name}")
print(f"Transform output: {transform_output}")


In [ ]:
# Download and inspect the batch output.
s3 = boto_session.client("s3", region_name=region)
output_key = f"zembed-1/batch-output/{suffix}/{batch_input_filename}.out"
local_output = "zembed_batch_output.jsonl.out"

s3.download_file(bucket, output_key, local_output)
print(Path(local_output).read_text()[:2000])


## 4. Clean-up


### A. Delete the model


In [ ]:
# Delete the SageMaker model after both endpoint and batch examples are finished.
try:
    sm_client.delete_model(ModelName=model_name)
    print(f"Deleted model: {model_name}")
except ClientError as exc:
    if exc.response["Error"].get("Code") == "ValidationException":
        print(f"Model was already deleted or does not exist: {model_name}")
    else:
        raise


### B. Unsubscribe to the listing (optional)


If you would like to unsubscribe from the model package, follow these steps. Before you cancel the subscription, ensure that you do not have any [deployable model](https://console.aws.amazon.com/sagemaker/home#/models) created from the model package or using the algorithm. You can find this information by looking at the container name associated with the model.

**Steps to unsubscribe from the product in AWS Marketplace**:
1. Navigate to the __Machine Learning__ tab on [__Your Software subscriptions page__](https://aws.amazon.com/marketplace/ai/library?productType=ml&ref_=mlmp_gitdemo_indust)
2. Locate the listing that you want to cancel, and then choose __Cancel Subscription__.


## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/aws_marketplace|curating_aws_marketplace_listing_and_sample_notebook|ModelPackage|Sample_Notebook_Template|zembed-1-SageMaker-model.ipynb)
